# Preprocesamiento de datos

En este notebook se describe el preprocesamiento aplicado al dataset para predecir el reingreso hospitalario a 30 días. Se escogió ese umbral porque es el estandar más habitual en la literatura.

Las principales tareas son:

- Limpieza de datos
- Tratamiento de valores faltantes
- Codificación de variables categóricas
- Transformación de variables
- Preparación del dataset final para modelado

Para ello se creó el script `preprocessing.py` con funciones reutilizables. El notebook sirve para comprobar paso a paso que todo funciona bien y justificar las decisiones tomadas.

En el pipeline se usan tres tablas de MIMIC-IV: `patients`, `admissions` y `diagnoses_icd`. A partir de ellas se construye el dataset base con información demográfica, datos de cada hospitalización y el número de diagnósticos por ingreso (`n_diagnoses`).

Las tablas de `procedures` y `prescriptions` no se incluyen en esta versión. Aunque se analizaron en el EDA, son muy grandes y su cardinalidad requeriría un feature engineering adicional que se deja para versiones futuras.

El preprocesamiento tiene dos etapas.

En la **primera** (`run_preprocessing_part1`) se limpia la estructura del dataset: se eliminan duplicados, se convierten las fechas a datetime, se calcula `length_of_stay` y se filtran registros con duración negativa. También se une la información de las tres tablas y se añade `n_diagnoses`.

En la **segunda** (`run_preprocessing_part2`) se hacen las transformaciones para el modelado:

- **Filtrado de fallecidos**: los pacientes con `hospital_expire_flag=1` se eliminan, ya que no pueden tener readmisión y sesgarían la distribución.
- **Variable objetivo** `readmission_30_days`: indica si el paciente reingresó en los 30 días siguientes. También se añade `previous_admissions`.
- **Edad real** (`age_at_admission`): MIMIC-IV desplaza las fechas por privacidad, así que `anchor_age` no da la edad exacta en cada ingreso. Se corrige como `anchor_age + (año_admisión - anchor_year)`.
- **Eliminación de variables** no útiles, con riesgo de fuga de información o redundantes.
- **Imputación** de categóricas con `Unknown`.
- **Agrupación** de categorías raras en `Other` (umbral 1%) para `race` y `language`.
- **Codificación one-hot** con `drop_first=True`.

# 1. Limpieza estructural y construcción del dataset base

En esta primera parte se comprueba que `run_preprocessing_part1` funciona bien y genera las variables derivadas que se necesitan.

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.data.preprocessing import  run_preprocessing_part1

df = run_preprocessing_part1()

df.head()

Loading datasets...
Loading patients...
Loading admissions...
Loading diagnoses...
Cleaning datasets...
Merging datasets...
Adding diagnosis features...
Computing previous admissions...
Saving interim dataset...
Preprocessing completed.


,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,edouttime,hospital_expire_flag,length_of_stay,gender,anchor_age,anchor_year,anchor_year_group,dod,n_diagnoses,previous_admissions
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,2180-05-06 23:30:00,0,0,F,52,2180,2014 - 2016,2180-09-09,8,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,...,2180-06-26 21:31:00,0,1,F,52,2180,2014 - 2016,2180-09-09,8,1
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,...,2180-07-23 14:00:00,0,2,F,52,2180,2014 - 2016,2180-09-09,13,2
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,...,2180-08-06 01:44:00,0,1,F,52,2180,2014 - 2016,2180-09-09,10,3
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,...,2160-03-04 06:26:00,0,0,F,19,2160,2008 - 2010,NaN,1,0


In [ ]:
# comprobamos que n_diagnoses se ha generado bien
print("n_diagnoses generado:", "n_diagnoses" in df.columns)
print(df["n_diagnoses"].describe())
print("Valores faltantes en n_diagnoses:", df["n_diagnoses"].isnull().sum())

In [3]:
df.shape


(545853, 24)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 545853 entries, 0 to 545852
Data columns (total 24 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   subject_id            545853 non-null  int64         
 1   hadm_id               545853 non-null  int64         
 2   admittime             545853 non-null  datetime64[ns]
 3   dischtime             545853 non-null  datetime64[ns]
 4   deathtime             11689 non-null   object        
 5   admission_type        545853 non-null  object        
 6   admit_provider_id     545849 non-null  object        
 7   admission_location    545852 non-null  object        
 8   discharge_location    396100 non-null  object        
 9   insurance             536519 non-null  object        
 10  language              545082 non-null  object        
 11  marital_status        532260 non-null  object        
 12  race                  545853 non-null  object        
 13  edre

In [5]:
df.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,edouttime,hospital_expire_flag,length_of_stay,gender,anchor_age,anchor_year,anchor_year_group,dod,n_diagnoses,previous_admissions
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,2180-05-06 23:30:00,0,0,F,52,2180,2014 - 2016,2180-09-09,8,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,...,2180-06-26 21:31:00,0,1,F,52,2180,2014 - 2016,2180-09-09,8,1
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,...,2180-07-23 14:00:00,0,2,F,52,2180,2014 - 2016,2180-09-09,13,2
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,...,2180-08-06 01:44:00,0,1,F,52,2180,2014 - 2016,2180-09-09,10,3
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,...,2160-03-04 06:26:00,0,0,F,19,2160,2008 - 2010,NaN,1,0


In [6]:
df.isnull().sum().sort_values(ascending=False)

deathtime               534164
dod                     400997
edouttime               166757
edregtime               166757
discharge_location      149753
marital_status           13593
insurance                 9334
language                   771
admit_provider_id            4
admission_location           1
admission_type               0
dischtime                    0
subject_id                   0
hadm_id                      0
admittime                    0
race                         0
hospital_expire_flag         0
length_of_stay               0
anchor_age                   0
gender                       0
anchor_year                  0
anchor_year_group            0
n_diagnoses                  0
previous_admissions          0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df["length_of_stay"].describe()

count    545853.000000
mean          4.222706
std           7.201838
min           0.000000
25%           1.000000
50%           2.000000
75%           5.000000
max         515.000000
Name: length_of_stay, dtype: float64

In [9]:
df["subject_id"].nunique()

223382

In [10]:
import os
os.listdir("../data/interim")

['.gitkeep', 'cleaned_dataset.csv']

In [11]:
summary = {
    "n_rows": df.shape[0],
    "n_columns": df.shape[1],
    "n_patients": df["subject_id"].nunique(),
    "n_admissions": df["hadm_id"].nunique()
}

summary

{'n_rows': 545853,
 'n_columns': 24,
 'n_patients': 223382,
 'n_admissions': 545853}

# 2. Preparación del conjunto final para modelado

En esta segunda parte se aplican las transformaciones finales sobre el dataset limpio. Trabajar desde el dataset intermedio evita repetir el procesamiento inicial cada vez.

## Variable objetivo y variables derivadas

La variable objetivo `readmission_30_days` se construye mirando si el paciente tuvo otro ingreso en los 30 días siguientes al alta.

Las últimas admisiones de cada paciente se eliminan porque no sabemos si hubo readmisión posterior, lo que introduciría sesgo.

Además se calculan dos variables derivadas:
- `previous_admissions`: cuántos ingresos previos tiene el paciente. Se calcula en la **primera etapa** del preprocesamiento (antes de filtrar fallecidos), para incluir también los ingresos con fallecimiento en el contador.
- `age_at_admission`: edad real en el momento del ingreso, corrigiendo el desplazamiento temporal de MIMIC-IV.

# Eliminación de variables e imputación

Antes de entrenar hay que decidir qué variables se quedan y cuáles se eliminan. Se excluyen las que:

- no aportan información útil para la predicción,
- tienen riesgo de fuga de información,
- son redundantes con variables ya derivadas,
- o son demasiado específicas para el objetivo.

Las variables categóricas con nulos se imputan con `Unknown` para no perder registros.

Variables como `deathtime`, `dod`, `edregtime` o `edouttime` se eliminan porque no son adecuadas para este problema predictivo.

Las variables temporales originales (`admittime`, `dischtime`) también se eliminan una vez que ya tenemos `length_of_stay`.

Para las categóricas con nulos (`insurance`, `marital_status`, `language`) se usa `Unknown` para conservar los registros sin distorsionar el modelo.

# Codificación de variables categóricas

Una vez imputados los nulos, las variables categóricas se codifican en *one-hot*. Así se representan sin imponer un orden artificial entre las categorías.

# Validación del pipeline implementado en `run_preprocessing_part2`


In [12]:
import sys
import os
import pandas as pd 
sys.path.append(os.path.abspath(".."))
from src.config import DATA_INTERIM
from src.data.preprocessing import run_preprocessing_part2


df =pd.read_csv(DATA_INTERIM/"cleaned_dataset.csv",
     parse_dates=["admittime","dischtime"]
     )
df.columns

Index(['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime',
       'admission_type', 'admit_provider_id', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status', 'race',
       'edregtime', 'edouttime', 'hospital_expire_flag', 'length_of_stay',
       'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod',
       'n_diagnoses', 'previous_admissions'],
      dtype='object')

In [13]:
df_model = run_preprocessing_part2(df.copy())
print("Columnas:", len(df_model.columns))

Preparing dataset for modeling...
Filtering in-hospital deaths...
Creating readmission target variable...
Columns before processing: 26
Columns after encoding: 56
Saving processed dataset...
Model dataset created.
Columnas: 56


In [14]:
df_model[["previous_admissions","readmission_30_days"]].head(10)

,previous_admissions,readmission_30_days
0,0,0
1,1,1
2,2,1
5,0,0
8,0,0
14,0,0
16,0,0
17,1,0
18,2,0
22,0,1


In [15]:
df_model["readmission_30_days"].value_counts()
df_model.head()

,subject_id,length_of_stay,n_diagnoses,previous_admissions,readmission_30_days,age_at_admission,gender_M,race_ASIAN - CHINESE,race_BLACK/AFRICAN AMERICAN,race_BLACK/CAPE VERDEAN,...,admission_location_PROCEDURE SITE,admission_location_TRANSFER FROM HOSPITAL,admission_location_TRANSFER FROM SKILLED NURSING FACILITY,admission_location_WALK-IN/SELF REFERRAL,discharge_location_HOME,discharge_location_HOME HEALTH CARE,discharge_location_Other,discharge_location_REHAB,discharge_location_SKILLED NURSING FACILITY,discharge_location_Unknown
0,10000032,0,8,0,0,52,False,False,False,False,...,False,True,False,False,True,False,False,False,False,False
1,10000032,1,8,1,1,52,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
2,10000032,2,13,2,1,52,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
5,10000084,4,6,0,0,72,True,False,False,False,...,False,False,False,True,False,True,False,False,False,False
8,10000117,0,9,0,0,55,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [16]:
len(df_model.columns)


56

In [ ]:
# verificamos que age_at_admission está bien calculada y en rango razonable
print("age_at_admission generado:", "age_at_admission" in df_model.columns)
print(df_model[["age_at_admission", "length_of_stay", "n_diagnoses", "previous_admissions"]].describe())

## Resumen del preprocesamiento

El dataset final tiene:

- **315.982 registros** (excluidos fallecidos en hospital y últimas admisiones sin seguimiento)
- **Variables numéricas**: `length_of_stay`, `age_at_admission`, `n_diagnoses`, `previous_admissions`
- **Variable objetivo**: `readmission_30_days`
- **Variables categóricas**: codificadas en one-hot, agrupando categorías raras al 1%
- **Total de columnas**: 56 tras el encoding

Se guarda en `data/processed/model_dataset.csv` y es el punto de entrada para el modelado.